In [ ]:
import pandas as pd
import numpy as np
import sqlalchemy

In [ ]:
import pandas as pd
import sqlite3

# Define the SQLite database file path
sqlite_db_path = '../DB/NEWDB/zomato_db_new.sqlite'

# Define the table name for the staging data
staging_table_name = 'staging_data'

# Connect to the SQLite database and retrieve the staging data
with sqlite3.connect(sqlite_db_path) as conn:
    # Query the staging data
    query = f"SELECT * FROM {staging_table_name}"
    staging_df = pd.read_sql(query, conn)

# Display the first 5 rows of the DataFrame
staging_df.head(5)


In [ ]:
staging_df.shape

In [ ]:
staging_df.info()

In [ ]:
staging_df.isna().sum()

In [ ]:
staging_df.info()  # Check data types and missing values
staging_df.describe()  # Summary statistics for numerical columns


In [ ]:
staging_df.drop_duplicates(inplace=True)


Distribution of Ratings

In [ ]:
staging_df['RATING'].hist(bins=10)


checking the correlation between the  Rating type and rating 

In [ ]:
from sklearn.preprocessing import LabelEncoder

#Label encode the 'RATING_TYPE' column
label_encoder = LabelEncoder()
staging_df['RATING_TYPE_ENCODED'] = label_encoder.fit_transform(staging_df['RATING_TYPE'])

# Step 2: Calculate the correlation between 'RATING' and 'RATING_TYPE_ENCODED'
correlation = staging_df[['RATING', 'RATING_TYPE_ENCODED']].corr()

# Display the correlation matrix
print(correlation)



Finding the missing rows distribution

In [ ]:

missing_both = staging_df['RATING'].isnull() & staging_df['RATING_TYPE'].isnull()

missing_both_count = missing_both.sum()


missing_rating_count = staging_df['RATING'].isnull().sum()
missing_rating_type_count = staging_df['RATING_TYPE'].isnull().sum()


print(f"Number of rows where both 'RATING' and 'RATING_TYPE' are missing: {missing_both_count}")
print(f"Number of rows where 'RATING' is missing: {missing_rating_count}")
print(f"Number of rows where 'RATING_TYPE' is missing: {missing_rating_type_count}")


The correlation matrix is good as the value is .8    so we can map the values each other  using linear regression

In [ ]:
from sklearn.linear_model import LinearRegression

import numpy as np

# Step 1: Label encode 'RATING_TYPE'
label_encoder = LabelEncoder()
staging_df['RATING_TYPE_ENCODED'] = label_encoder.fit_transform(staging_df['RATING_TYPE'].astype(str))

# Step 2: Identify missing values
missing_rating = staging_df['RATING'].isnull()
missing_rating_type = staging_df['RATING_TYPE'].isnull()


both_missing = missing_rating & missing_rating_type


X = staging_df[['RATING_TYPE_ENCODED']][~missing_rating & ~missing_rating_type] 
y = staging_df['RATING'][~missing_rating & ~missing_rating_type]  

model = LinearRegression()
model.fit(X, y)


staging_df.loc[missing_rating & ~missing_rating_type, 'RATING'] = model.predict(staging_df[['RATING_TYPE_ENCODED']][missing_rating & ~missing_rating_type])




rating_type_mode = staging_df.groupby('RATING')['RATING_TYPE'].agg(lambda x: x.mode()[0] if len(x.mode()) > 0 else np.nan)

staging_df.loc[missing_rating_type & ~missing_rating, 'RATING_TYPE'] = staging_df.loc[missing_rating_type & ~missing_rating, 'RATING'].map(rating_type_mode)


print(staging_df.isnull().sum())


1. Which locality lists the maximum number of hotels?

In [ ]:
import matplotlib.pyplot as plt

# Group by 'REGION' (locality) and count the number of hotels
region_counts = staging_df['REGION'].value_counts()

# Plot the results
plt.figure(figsize=(10, 6))
region_counts.head(10).plot(kind='bar', color='skyblue')
plt.title('Top 10 Localities by Number of Hotels')
plt.xlabel('Locality')
plt.ylabel('Number of Hotels')
plt.xticks(rotation=45)
plt.show()


2.What cuisine is served in the highest-rated hotels in the country?

In [ ]:
# Filtering the hotels with a high rating (e.g., above 4.5)
high_rated_hotels = staging_df[staging_df['RATING'] >= 4.5]

cuisine_counts = high_rated_hotels['CUSINE_CATEGORY'].value_counts()

plt.figure(figsize=(10, 6))
cuisine_counts.head(10).plot(kind='bar', color='green')
plt.title('Top Cuisines in Highest Rated Hotels')
plt.xlabel('Cuisine Type')
plt.ylabel('Number of Hotels')
plt.xticks(rotation=45)
plt.show()


3. Is there any relation between the cost of dining and the hotel rating?

In [ ]:
# Scatter plot to show the relationship between 'PRICE' and 'RATING'
plt.figure(figsize=(8, 6))
plt.scatter(staging_df['PRICE'], staging_df['RATING'], color='purple', alpha=0.5)
plt.title('Relation Between Price and Rating')
plt.xlabel('Price of Dining')
plt.ylabel('Hotel Rating')
plt.show()


In [ ]:
import pandas as pd

# Calculate the correlation coefficient between PRICE and RATING
correlation = staging_df['PRICE'].corr(staging_df['RATING'])

# Print the result
print(f"The correlation coefficient between Price and Rating is: {correlation}")



Observation:As the Correlation coefficient is very low there is little to no realtion between the price and rating

6. Which cuisine dominates Indian taste buds and is there any relation to where the hotel is located?

In [ ]:
staging_df.head(5)